## **Target Encoding**
### **Introduction**
- Suitable in a supervised setting where categorical feature $X$ is used to predict some output $y$.
- Preferred for tree-based models.
- Encode the categorical value $c$ as,
$$ \text{Enc}(c) = \mathbb{E}[ y\;|\;X=c] $$
### **Motivation**
- Suitable for features with high cardinality (like cities, zip codes, user ids etc.), where OHE introduces too many columns.
- Replaces arbitrary labels with the actual statistical relationship between the category and the target variable.
- Allows Gradient Boosted Trees (XGBoost, CatBoost) to find optimal splits in a single step rather than traversing deep.

In [6]:
import pandas as pd
import numpy as np

In [7]:
n_samples = 1000
city_list = ['New York', 'Los Angeles', 'Chicago']

# sales somehow depends on the city
def get_sales(city):
    match city:
        case 'New York':
            return np.random.randint(500, 1000)
        case 'Los Angeles':
            return np.random.randint(300, 800)
        case 'Chicago':
            return np.random.randint(100, 600)
        case _:
            raise KeyError(f"Unknown city: {city}")

cities = np.random.choice(city_list, n_samples)
sales = [get_sales(city) for city in cities]

data = pd.DataFrame({
    'City': cities,
    'Sales': sales
})

data.head()

,City,Sales
0,Los Angeles,698
1,Chicago,501
2,New York,749
3,Los Angeles,602
4,Chicago,371


In [8]:
from category_encoders import TargetEncoder

encoder = TargetEncoder()
data_TE = data.copy()
data_TE['City_TE'] = encoder.fit_transform(X=data_TE['City'], y=data_TE['Sales'])
data_TE.head()

,City,Sales,City_TE
0,Los Angeles,698,548.621451
1,Chicago,501,345.579545
2,New York,749,760.580060
3,Los Angeles,602,548.621451
4,Chicago,371,345.579545


### **Challenges**
- *Target Leakage:* Since the encoding uses the target value, the model can "see" the answer during training.
- *High Variance for small categories:* Small categories with few samples can result in extreme, unreliable mean values that don't generalize.
- *Distribution shift (train-test mismatch):* Target encoding creates a feature that is highly specific to the training set's distribution.

### **Bayesian Target Encoding**
Addresses the instability of small categories by shrinking the category mean toward the overall global mean of the dataset.\
Encode the categorical value $c$ as,
$$ \text{Enc}(c) = \frac{n_c \times \mu_c + \alpha \times \mu}{n_c + \alpha} $$
where:\
$\mu_c = \mathbb{E}[y|X=c]$ \
$\mu = \mathbb{E}[y]$\
$n_c = $ number of times the category $c$ occurs in the dataset\
$\alpha = $ smoothing factor (hyperparameter)

In [17]:
from category_encoders import MEstimateEncoder

encoder = MEstimateEncoder(m=10)
data_BTE = data.copy()
data_BTE['City_BTE'] = encoder.fit_transform(X=data_BTE['City'], y=data_BTE['Sales'])
data_BTE.head()

,City,Sales,City_BTE
0,Los Angeles,698,548.581315
1,Chicago,501,351.152182
2,New York,749,754.325777
3,Los Angeles,602,548.581315
4,Chicago,371,351.152182


### **Things to keep in mind**
- *Leave-One-Out Encoding:* When calculating the mean for a specific row, exclude that row's own target value from the calculation.
- *K-Fold Encodings:* Split your training data into folds. Calculate the encoding for "Fold A" using only the data from "Folds B, C, and D." This ensures no row "sees" its own target.
- *Add Random Noise:* Add a tiny amount of Gaussian noise to the final encoded values. This prevents the model from mapping exact floating-point values to specific targets.

Some other important techniques for encoding from category_encoder library include,
1) `LeaveOneOutEncoder`
A variant of target encoding that excludes the current row's target value when computing the category mean — reduces overfitting compared to plain target encoding.
  
2) `WOEEncoder`
Encodes categories using the log-ratio of the probability of the target being 1 vs 0 — widely used in credit scoring and binary classification problems in finance.
  
3) `JamesSteinEncoder`
Uses the James-Stein shrinkage estimator to encode categories — reduces variance by shrinking category estimates toward the global mean, works well with small datasets.
  
4) `QuantileEncoder`
Replaces each category with a quantile statistic (e.g. median) of the target variable, making it more robust to outliers than standard target encoding.
  
5)  `PolynomialEncoder`
Applies polynomial contrast coding to capture linear, quadratic, and higher-order trends across ordered categories — best suited for ordinal data in statistical/regression models.
